In [1]:
from collections import defaultdict
import cv2
import numpy as np
from ultralytics import YOLO
import yt_dlp
import math
import time

# Analytics variables
behavior_history = defaultdict(list)
speed_history = defaultdict(list)
behavior_counts = defaultdict(int)
session_start = time.time()


def get_youtube_stream_url(youtube_url):
    ydl_opts = {"format": "best[height<=720]", "quiet": True, "no_warnings": True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
        return info["url"]


def classify_behavior(fish_id, current_pos, previous_positions):
    if len(previous_positions) < 3:
        return "initializing"

    x1, y1 = previous_positions[-2]
    x2, y2 = previous_positions[-1]
    speed = math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

    speed_history[fish_id].append(speed)
    if len(speed_history[fish_id]) > 10:
        speed_history[fish_id].pop(0)

    avg_speed = sum(speed_history[fish_id]) / len(speed_history[fish_id])

    if avg_speed < 2:
        return "resting"
    elif avg_speed > 10:
        return "active"
    elif avg_speed > 5:
        return "swimming"
    else:
        return "cruising"


def print_stats():
    session_time = time.time() - session_start
    total = sum(behavior_counts.values())
    print(f"\n📊 Session: {session_time:.1f}s | Behaviors: {total}")
    for behavior, count in behavior_counts.items():
        percentage = (count / total) * 100 if total > 0 else 0
        print(f"   {behavior}: {count} ({percentage:.1f}%)")


def get_insights():
    total = sum(behavior_counts.values())
    if total < 10:
        return ["📈 Collecting data..."]

    insights = []
    most_common = max(behavior_counts, key=behavior_counts.get)
    percentage = (behavior_counts[most_common] / total) * 100
    insights.append(f"🎯 Most common: {most_common} ({percentage:.1f}%)")

    active = behavior_counts.get("active", 0) + behavior_counts.get("swimming", 0)
    if active > total * 0.6:
        insights.append("🏃 Very active fish!")
    elif behavior_counts.get("resting", 0) > total * 0.5:
        insights.append("😴 Fish are resting")

    return insights


# Main execution
custom_model_file = "./yolov8_runs/train/weights/best.pt"
model = YOLO(custom_model_file)
youtube_url = "https://youtu.be/KLPApSRku8Y?si=BoYvbjxc744krjZW"

try:
    stream_url = get_youtube_stream_url(youtube_url)
    cap = cv2.VideoCapture(stream_url)
    if not cap.isOpened():
        exit("Could not open stream")
except Exception as e:
    exit(f"Error: {e}")

track_history = defaultdict(lambda: [])
last_stats = time.time()
last_insights = time.time()

print("🐟 Fish Analytics Started! Press 'q' to quit, 's' for stats")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    result = model.track(frame, persist=True)[0]

    if result.boxes and result.boxes.is_track:
        boxes = result.boxes.xywh.cpu()
        track_ids = result.boxes.id.int().cpu().tolist()
        frame = result.plot()

        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            current_pos = (float(x), float(y))

            track = track_history[track_id]
            track.append(current_pos)
            if len(track) > 30:
                track.pop(0)

            # Behavior classification
            behavior = classify_behavior(track_id, current_pos, track)
            behavior_counts[behavior] += 1

            # Display behavior
            cv2.putText(
                frame,
                f"Fish {track_id}: {behavior}",
                (int(x - w / 2), int(y - h / 2 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2,
            )

    cv2.imshow("🐟 Fish Analytics", frame)

    # Auto stats every 5 seconds
    if time.time() - last_stats > 5:
        print_stats()
        last_stats = time.time()

    # Auto insights every 10 seconds
    if time.time() - last_insights > 10:
        print("\n🔍 Insights:")
        for insight in get_insights():
            print(f"   {insight}")
        last_insights = time.time()

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break
    elif key == ord("f"):  # Forward skip
        current_frame = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        skip_frames = int(30 * fps)  # 30 seconds worth of frames
        new_frame = min(current_frame + skip_frames, frame_count)
        cap.set(cv2.CAP_PROP_POS_FRAMES, new_frame)
        print(f"Skipped forward to frame {new_frame}")

    elif key == ord("b"):  # Backward skip
        current_frame = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        skip_frames = int(30 * fps)  # 30 seconds worth of frames
        new_frame = max(current_frame - skip_frames, 0)
        cap.set(cv2.CAP_PROP_POS_FRAMES, new_frame)
        print(f"Skipped backward to frame {new_frame}")

print("\n🎯 Final Analytics:")
print_stats()
print("\n🔍 Final Insights:")
for insight in get_insights():
    print(f"   {insight}")

cap.release()
cv2.destroyAllWindows()

🐟 Fish Analytics Started! Press 'q' to quit, 's' for stats

0: 384x640 (no detections), 35.7ms
Speed: 2.4ms preprocess, 35.7ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fish, 32.5ms
Speed: 0.8ms preprocess, 32.5ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fish, 66.8ms
Speed: 0.9ms preprocess, 66.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fish, 40.8ms
Speed: 1.2ms preprocess, 40.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fish, 35.5ms
Speed: 1.0ms preprocess, 35.5ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fish, 33.1ms
Speed: 0.8ms preprocess, 33.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fish, 32.7ms
Speed: 0.7ms preprocess, 32.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 fish, 33.1ms
Speed: 0.8ms preprocess, 33.1ms infer